In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('../data/processed/train_processed.csv')
print(train_df.shape)
train_df.head()
print(train_df.columns)

(20631, 20)
Index(['engine_id', 'op_setting_1', 'op_setting_2', 'sensor_2', 'sensor_3',
       'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17',
       'sensor_20', 'sensor_21', 'cycle_norm', 'RUL'],
      dtype='object')


Doing train validation split by engine_id

In [2]:
# %%
from sklearn.model_selection import train_test_split

# unique engine ids
engine_ids = train_df['engine_id'].unique()

# split by engine
train_ids, val_ids = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42
)

# %%
# filtering train and validation engines
train_data = train_df[train_df['engine_id'].isin(train_ids)].copy()

val_data = train_df[train_df['engine_id'].isin(val_ids)].copy()

print("Train engines:", len(train_ids))
print("Validation engines:", len(val_ids))


Train engines: 80
Validation engines: 20


Doing feature scaling now (only on training data)

In [3]:
from sklearn.preprocessing import StandardScaler

# feature columns
feature_cols = [
    col for col in train_df.columns
    if col not in ['engine_id', 'RUL']
]

# scaler
scaler = StandardScaler()

# fit ONLY on train
train_data[feature_cols] = scaler.fit_transform(
    train_data[feature_cols]
)

# transform validation
val_data[feature_cols] = scaler.transform(
    val_data[feature_cols]
)

# %%
print(train_data.shape)
print(val_data.shape)

(16561, 20)
(4070, 20)


Creating Sequences

In [4]:
def create_sequences(df, seq_length, feature_cols):

    sequences = []
    labels = []

    for engine_id in df['engine_id'].unique():

        engine_data = df[df['engine_id'] == engine_id]

        data = engine_data[feature_cols].values
        rul = engine_data['RUL'].values

        for i in range(len(data) - seq_length + 1):

            seq = data[i:i + seq_length]

            label = rul[i + seq_length - 1]

            sequences.append(seq)
            labels.append(label)

    return np.array(sequences), np.array(labels)

# %%
sequence_length = 30

# %%
X_train, y_train = create_sequences(
    train_data,
    sequence_length,
    feature_cols
)

X_val, y_val = create_sequences(
    val_data,
    sequence_length,
    feature_cols
)

# %%
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

# %%
print(X_train[0].shape)
print(y_train[0])

X_train shape: (14241, 30, 18)
y_train shape: (14241,)
X_val shape: (3490, 30, 18)
y_val shape: (3490,)
(30, 18)
125


Now doing the test/train split

note: 
I am doing engine wise split

In [5]:
# %%
# original raw column names
columns = ['engine_id', 'cycle'] + \
          [f'op_setting_{i}' for i in range(1, 4)] + \
          [f'sensor_{i}' for i in range(1, 22)]

# %%
test_df = pd.read_csv(
    '../CMAPSSData/test_FD001.txt',
    sep=' ',
    header=None
)

# remove extra empty columns
test_df = test_df.dropna(axis=1)

# assign names
test_df.columns = columns

# %%
print(test_df.shape)

(13096, 26)


In [6]:
# %% [markdown]
# Apply SAME preprocessing as training pipeline

# %%
# drop same constant/useless columns
drop_cols = [
    'sensor_1',
    'sensor_5',
    'sensor_10',
    'sensor_16',
    'sensor_18',
    'sensor_19',
    'op_setting_3'
]

test_df = test_df.drop(columns=drop_cols)

# compute from training data
mean_max_cycle = 206.31

# apply to test
test_df['cycle_norm'] = test_df.groupby('engine_id')['cycle'].transform(
    lambda x: x / mean_max_cycle
)
# %%
# drop raw cycle column
test_df = test_df.drop(columns=['cycle'])

# %%
# apply SAME scaler (transform only)
test_df[feature_cols] = scaler.transform(
    test_df[feature_cols]
)

# %%
print(test_df.shape)

(13096, 19)


In [7]:
# %% [markdown]
# Creating Test Sequences
# IMPORTANT:
# for test set we use ONLY the last sequence of each engine

# %%
def get_last_sequences(df, seq_length, feature_cols):

    sequences = []

    for engine_id in df['engine_id'].unique():

        engine_data = df[df['engine_id'] == engine_id]

        data = engine_data[feature_cols].values

        last_sequence = data[-seq_length:]

        sequences.append(last_sequence)

    return np.array(sequences)

# %%
X_test_final = get_last_sequences(
    test_df,
    sequence_length,
    feature_cols
)

# %%
print("X_test_final shape:", X_test_final.shape)


X_test_final shape: (100, 30, 18)


In [ ]:
# %% [markdown]
# Loading True Test RUL

# %%
rul_test = pd.read_csv(
    '../CMAPSSData/RUL_FD001.txt',
    header=None
)

# clip at 125 (same as train)
rul_test = np.clip(rul_test.values, 0, 125)

# %%
print("rul_test shape:", rul_test.shape)

# %% [markdown]

X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test_final = X_test_final.astype(np.float32)

y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
rul_test = rul_test.astype(np.float32)
# Saving Processed Arrays

# %%
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/y_train.npy', y_train)

np.save('../data/processed/X_val.npy', X_val)
np.save('../data/processed/y_val.npy', y_val)

np.save('../data/processed/X_test_final.npy', X_test_final)
np.save('../data/processed/rul_test.npy', rul_test)



# %%
print("All processed files saved successfully.")

rul_test shape: (100, 1)
All processed files saved successfully.


In [6]:
import joblib
joblib.dump(scaler, '../data/processed/scaler.pkl')

NameError: name 'scaler' is not defined